# 18 · 文档加载器选型 & 失败模式

> **学习目标**：用本机 [rag_project/docs/](../../../../rag_project/docs/) 里真实的中文 PDF / DOCX，对比 pypdfium2 / pypdf / python-docx 在「中文 / 含表格 / 含图」场景下的差异。建立 loader 选型决策表。
>
> **预备**：16、17 跑过。
>
> **为什么重要**：**RAG 90% 的烂 demo 死在数据加载阶段**。中文 PDF 文字粘连、表格被切碎、扫描件没文字 …… 全是 loader 没选对、没兜底。

In [ ]:
from pathlib import Path
DOCS = Path('f:/source/code/direction/rag/rag_project/docs')
assert DOCS.exists(), 'rag_project/docs 不存在'

# 抓 1 个 PDF + 1 个 DOCX 用于演示
pdfs  = sorted(DOCS.rglob('*.pdf'))
docxs = [p for p in DOCS.rglob('*.docx') if not p.name.startswith('~$')]   # 跳 office lock
print(f'本机可用 PDF  : {len(pdfs)}  (示例：{pdfs[0].name if pdfs else None!r})')
print(f'本机可用 DOCX : {len(docxs)} (示例：{docxs[0].name if docxs else None!r})')

## 1. PDF：pypdfium2 vs pypdf

**两家定位差别**：
- `pypdfium2`：基于 Google PDFium（Chrome PDF 引擎），**中文 / 复杂版式更好**，原生 C++ 加持
- `pypdf`：纯 Python，**最易安装**，简单 PDF 够用，复杂版式可能掉字 / 乱序

**实测策略**：同一个 PDF 用两家分别抽前 800 字符，肉眼对比。

In [ ]:
import pypdfium2 as pdfium
import pypdf
import time

def load_with_pypdfium2(path: Path, max_pages: int = 3) -> str:
    doc = pdfium.PdfDocument(str(path))
    parts = []
    for i in range(min(max_pages, len(doc))):
        page = doc[i]
        text = page.get_textpage().get_text_range()
        parts.append(f'[PAGE {i+1}]\n{text}')
    return '\n\n'.join(parts)

def load_with_pypdf(path: Path, max_pages: int = 3) -> str:
    reader = pypdf.PdfReader(str(path))
    parts = []
    for i, page in enumerate(reader.pages[:max_pages]):
        parts.append(f'[PAGE {i+1}]\n{page.extract_text() or ""}')
    return '\n\n'.join(parts)

sample_pdf = pdfs[0]
print(f'样本 PDF: {sample_pdf.name}\n')

for name, fn in [('pypdfium2', load_with_pypdfium2), ('pypdf', load_with_pypdf)]:
    t0 = time.perf_counter()
    text = fn(sample_pdf, max_pages=2)
    dt = (time.perf_counter() - t0) * 1000
    preview = text[:300].replace('\n', ' | ')
    print(f'--- {name}  耗时 {dt:.1f} ms  抽出 {len(text)} 字符 ---')
    print(f'preview: {preview}...')
    print()

In [ ]:
# 健壮性套路：主用 pypdfium2，pypdf 兜底
def robust_pdf_load(path: Path) -> tuple[str, str]:
    """返回 (text, used_backend)。前者非空就用，否则换备份。"""
    try:
        text = load_with_pypdfium2(path, max_pages=999)
        if text.strip():
            return text, 'pypdfium2'
    except Exception as e:
        print(f'  pypdfium2 失败: {type(e).__name__}: {str(e)[:60]}')
    try:
        text = load_with_pypdf(path, max_pages=999)
        return text, 'pypdf' if text.strip() else 'EMPTY'
    except Exception as e:
        return '', f'BOTH_FAILED: {type(e).__name__}'

# 跑前 3 个 PDF 各测一遍
for p in pdfs[:3]:
    text, backend = robust_pdf_load(p)
    status = '✅' if text.strip() else '❌'
    print(f'{status} {p.name:60} [{backend}] 抽 {len(text):>6} 字符')

## 2. DOCX：python-docx —— 段落与表格要分开取

**关键点**：`Document.paragraphs` 只给正文段落；表格里的文字不会出现在 `paragraphs` 里。必须 **同时遍历 `tables`**，否则表格内容直接丢。

In [ ]:
from docx import Document

def load_docx_naive(path: Path) -> str:
    """只抓段落 —— 会漏表格"""
    doc = Document(str(path))
    return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())

def load_docx_complete(path: Path) -> str:
    """段落 + 表格都抓"""
    doc = Document(str(path))
    parts = ['# 正文段落']
    for p in doc.paragraphs:
        if p.text.strip():
            parts.append(p.text)
    for i, tbl in enumerate(doc.tables):
        parts.append(f'\n# 表格 {i+1}')
        for row in tbl.rows:
            cells = [c.text.strip().replace('\n', ' ') for c in row.cells]
            parts.append(' | '.join(cells))
    return '\n'.join(parts)

if docxs:
    sample_docx = docxs[0]
    print(f'样本 DOCX: {sample_docx.name}\n')
    naive_text = load_docx_naive(sample_docx)
    full_text  = load_docx_complete(sample_docx)
    print(f'naive（只段落）: {len(naive_text):>6} 字符')
    print(f'complete（含表）: {len(full_text):>6} 字符')
    print(f'\n→ 差值 {len(full_text) - len(naive_text)} 字符就是「表格里的内容」 —— 一旦忘抓直接丢。')
    print(f'\n--- complete 版本预览（前 400 字符）---')
    print(full_text[:400])
else:
    print('（无 DOCX 样本，跳过本节）')

## 3. Markdown / TXT —— 看似最简单，坑在「保留结构」

**为什么 RAG 里 Markdown 受欢迎**：保留 # 标题层级、列表、代码块边界，**chunk 时可以按结构切**（详见 notebook 19）。但纯 `path.read_text()` 会丢失「这是 H2、那是 code block」的元信息。

In [ ]:
import re

# 用本目录的 README.md 当样本
sample_md = Path('../README.md').resolve()
if not sample_md.exists():
    sample_md = Path('f:/source/code/direction/rag/learning-roadmap/01-RAG/practice/README.md')

raw = sample_md.read_text(encoding='utf-8')

# 朴素解析：按 # 标题切，保留层级
def parse_markdown_sections(text: str) -> list[dict]:
    sections = []
    current = {'title': '(intro)', 'level': 0, 'body': []}
    for line in text.splitlines():
        m = re.match(r'^(#{1,6})\s+(.+)$', line)
        if m:
            if current['body']:
                sections.append({**current, 'body': '\n'.join(current['body'])})
            current = {'title': m.group(2), 'level': len(m.group(1)), 'body': []}
        else:
            current['body'].append(line)
    if current['body']:
        sections.append({**current, 'body': '\n'.join(current['body'])})
    return sections

sects = parse_markdown_sections(raw)
print(f'{sample_md.name} 解析出 {len(sects)} 个 section:')
for s in sects[:6]:
    print(f'  H{s["level"]}  {s["title"][:40]:40}  ({len(s["body"])} chars)')
if len(sects) > 6:
    print(f'  ... 还有 {len(sects) - 6} 个')

## 4. 失败模式合集 —— 这 5 种坑见一次记一辈子

**全是真实生产案例**。

In [ ]:
failures = [
    {
        'mode': '扫描件 PDF',
        '症状': 'load 后 text 几乎为空（< 50 字符）',
        '原因': '页面是图像不是文字，pypdf / pypdfium2 都抽不出字',
        '对策': '检测「文本字数 < 阈值 × 页数」时切到 OCR：PaddleOCR / Tesseract / MinerU',
    },
    {
        'mode': '表格被压平',
        '症状': '"型号  规格  单价\\n A B 100" 全粘在一行，列对应关系丢失',
        '原因': 'PDF 渲染时表格的物理位置被 extract_text 序列化按 y → x 顺序变成纯字符流',
        '对策': '用 camelot / pdfplumber / unstructured 等专门的 table extractor，输出 Markdown 表',
    },
    {
        'mode': 'Office 临时锁文件',
        '症状': '加载 ~$xxx.xlsx 报错',
        '原因': 'Word/Excel 打开文件时会留 ~$ 前缀 lock 文件，loader 不能跳过就崩',
        '对策': '在 loader 入口过滤 `name.startswith("~$")`（rag_project 的 default.yaml 已配）',
    },
    {
        'mode': '中文 PDF 字符乱码',
        '症状': '抽出 "￿￿￿￿" 或 "中国" 变 "\u4e2d\u56fd" 类乱码',
        '原因': 'PDF 内嵌的中文字体没有 ToUnicode 映射（特别是早期 PDFCreator 出的）',
        '对策': 'pypdfium2 比 pypdf 兼容性强；实在不行 OCR 兜底',
    },
    {
        'mode': '图片纯图无元数据',
        '症状': '加载 .png .jpg 直接没有 text',
        '原因': '图像本身没有文字层 —— 这跟扫描件本质相同',
        '对策': 'OCR；或用 vision LLM（如 Qwen-VL）「看图说话」生成描述再 embed',
    },
]
for f in failures:
    print(f'\n● {f["mode"]}')
    for k, v in f.items():
        if k == 'mode': continue
        print(f'  {k}: {v}')

In [ ]:
# 实战：扫描件检测器 —— 自动判断「这个 PDF 是不是基本没文字」
def looks_like_scanned_pdf(path: Path, threshold_per_page: int = 50) -> bool:
    try:
        doc = pdfium.PdfDocument(str(path))
        if len(doc) == 0:
            return False
        sample_pages = min(3, len(doc))
        total_chars = sum(len((doc[i].get_textpage().get_text_range() or '').strip()) for i in range(sample_pages))
        return total_chars / sample_pages < threshold_per_page
    except Exception:
        return False

print('扫描件检测（threshold = 50 字符/页）:')
for p in pdfs[:5]:
    scanned = looks_like_scanned_pdf(p)
    print(f'  {"📷 疑似扫描" if scanned else "📝 文本 PDF"}   {p.name}')

## 5. 选型决策表

| 文档类型 | 首选 loader | 何时切到备份 | 拒答场景 |
|---------|-------------|--------------|---------|
| 纯文本 PDF（论文 / 规范） | `pypdfium2` | 失败时退 `pypdf` | 加密 PDF 直接跳过 |
| 复杂版式 PDF（含图表） | `pypdfium2` + `pdfplumber` 提表 | 单纯抽文本会漏表 | 同上 |
| 扫描件 PDF | OCR（PaddleOCR / MinerU） | — | 中文扫描件**必须** OCR |
| DOCX | `python-docx`（段落 **+** 表格都遍历） | 表格大量时考虑转 PDF 再走 PDF 流水线 | 嵌入图片需另行 OCR |
| XLSX | `openpyxl` 读单元格；按表名分 Document | 含合并单元格时要预处理 | — |
| Markdown | 按 `#` 标题层级切 section | — | 大量代码块时按 ``` 边界切 |
| 代码（py / ts / java） | 按函数 / 类边界切，保留缩进 | tree-sitter 更准 | 不要按行强切 |
| 图片 / 截图 | 视觉模型（Qwen-VL / Pixtral）描述后 embed | OCR + vision 双路并 | — |

## 深入思考

1. **「主 + 备」loader 模式有什么副作用？**
   - 同一份 PDF 被解析两次（pypdfium2 + pypdf 各跑），慢；但生产 ingest 是离线一次性任务，慢一点没关系，**召回准比快重要**。
2. **为什么扫描件检测要按「字符/页」而不是「总字符」？**
   - 总字符受页数影响。50 字符/页是经验阈值（封面页 / 目录页正常情况下也至少有几百字符）。
3. **PDF 表格抽取到底哪家强？**
   - 没有银弹。`camelot` 适合规整线框表；`pdfplumber` 适合无线框；MinerU / unstructured 用 AI 模型，召回最高但慢。**先用一家跑，统计召回率不行再换**。
4. **Markdown 按 `#` 切，但章节里全是 1 行短笔记怎么办？**
   - section 太小直接合并（rolling window）。「按结构切到一定粒度，再二次切」是 langchain 的 MarkdownHeaderTextSplitter + RecursiveCharacterTextSplitter 的组合。
5. **当 loader 抽出 "乱码 + 真实文字混合" 时怎么办？**
   - 加一个「中文字符比例」过滤：`if zh_chars / total < 0.3: drop`，把乱码段先丢掉。**胜过让乱码污染向量库**。

**改一改**：写一个 walker，扫描 rag_project/docs 下所有文件，用 `robust_pdf_load` + `looks_like_scanned_pdf` 出一份「健康度报告」（哪些是文本 PDF、哪些是扫描件、哪些 loader 失败）。

## 自检 ✅

- [ ] 不查代码列出 PDF loader 两家（pypdfium2 / pypdf）的差异与各自适用场景。
- [ ] 解释「为什么 python-docx 只读 paragraphs 会丢表格」。
- [ ] 默背 5 种 loader 失败模式 + 对策。
- [ ] 给一份扫描件 PDF + 一份代码仓 README，能 5 分钟内决定各自该用什么 loader。
- [ ] 给一个 RAG 项目报告「召回不准」，能立刻问：「文档怎么加载的？让我看下 ingest 后的 chunk 样本」。

## 下一步

进入 Stage 2 → [`../stage2_进阶/19_chunking_strategies.ipynb`](../stage2_进阶/19_chunking_strategies.ipynb)